In [1]:
# 1. Instalar los paquetes de conversión (si ya están instalados, no hace nada)
#    - onnx / onnxruntime: el formato estándar y el motor que lo ejecuta (en el navegador será onnxruntime-web)
#    - tf2onnx: lo usa Keras 3 por dentro en model.export(..., format="onnx"). Se instala SIN dependencias
#      para no cambiar el numpy/protobuf que usa TensorFlow 2.21 (flatbuffers y requests ya están en el entorno)
%pip install onnx onnxruntime flatbuffers requests --quiet
%pip install tf2onnx --no-deps --quiet

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
import glob
import time
import random
import shutil
import tempfile
import tomllib
from types import SimpleNamespace

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow import keras
import onnx
import onnxruntime as ort
import tf2onnx
from treys import Card, Evaluator
from treys.lookup import LookupTable

# 1. Carpeta de salida: todo lo que necesita la app va en app/modelo_transformado/
SALIDA = "../app/modelo_transformado"
os.makedirs(SALIDA, exist_ok=True)
NOMBRES_CLASES = ['Fold', 'Check/Call', 'Bet/Raise']      # orden de las 3 salidas de TODOS los modelos
OPSET = 17

print("=" * 65)
print("  📖 NOTEBOOK 13 · EXPORTAR LOS MODELOS PARA LA APP WEB")
print("=" * 65)
print(f" • TensorFlow {tf.__version__} | Keras {keras.__version__} | tf2onnx {tf2onnx.__version__}")
print(f" • onnx {onnx.__version__} | onnxruntime {ort.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f" • Carpeta de salida: {os.path.abspath(SALIDA)}")

  📖 NOTEBOOK 13 · EXPORTAR LOS MODELOS PARA LA APP WEB
 • TensorFlow 2.21.0 | Keras 3.15.1 | tf2onnx 1.17.0
 • onnx 1.23.2 | onnxruntime 1.32.0 | numpy 2.5.5 | pandas 3.1.1
 • Carpeta de salida: C:\Users\lucas\Claude\PHT-T\app\modelo_transformado


In [3]:
def cargar_json(ruta):
    """Lee un archivo JSON en UTF-8 y devuelve su contenido."""
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

# 1. Las variables de los Modelos 3 y 4 salen de los JSON guardados junto a cada modelo
config3 = cargar_json("../models/v3_45_variables/variables_modelo3.json")
config4 = cargar_json("../models/v4_pluribus/variables_modelo4.json")
FEATURES_M1_M2 = ['calle', 'bote', 'to_call', 'stack', 'pot_odds']

# 2. variables_log se reduce a las que SÍ son entrada del modelo (el JSON también lista bote, to_call y stack, que no se usan; el código original las ignoraba)
# 3. Ficha de cada modelo: dónde está, cómo se preparó y con qué test se evalúa.
#    'pre_float32' = el entrenamiento convertía a float32 ANTES de escalar (Modelos 3 y 4: Notebooks 09 y 11)
MODELOS = {
    1: dict(nombre="Modelo 1 · con fuga (HandHQ, 5 variables)", carpeta="v1_con_fuga",
            keras="pht_t_modelo_poker.keras", scaler="scaler_poker.pkl",
            features=FEATURES_M1_M2, variables_log=[], pre_float32=False,
            test="../data/processed/modelo1/test.parquet", ref_exactitud=92.90),
    2: dict(nombre="Modelo 2 · corregido (HandHQ, 5 variables)", carpeta="v2_corregido",
            keras="pht_t_modelo_poker.keras", scaler="scaler_poker.pkl",
            features=FEATURES_M1_M2, variables_log=[], pre_float32=False,
            test="../data/processed/test.parquet", ref_exactitud=64.86),
    3: dict(nombre="Modelo 3 · 44 variables públicas (HandHQ)", carpeta="v3_45_variables",
            keras="pht_t_modelo3.keras", scaler="scaler_modelo3.pkl",
            features=config3['features'], variables_log=[c for c in config3['variables_log'] if c in config3['features']], pre_float32=True,
            test="../data/processed/modelo3/test.parquet", ref_exactitud=73.83),
    4: dict(nombre="Modelo 4 · Pluribus (47 variables con cartas)", carpeta="v4_pluribus",
            keras="pht_t_modelo4.keras", scaler="scaler_modelo4.pkl",
            features=config4['features'], variables_log=[c for c in config4['variables_log'] if c in config4['features']], pre_float32=True,
            test="../data/processed/modelo4/test.parquet", ref_exactitud=87.39),
}

print("=" * 65)
print("  🎯 LOS 4 MODELOS QUE SE EXPORTAN")
print("=" * 65)
for m, cfg in MODELOS.items():
    print(f" • Modelo {m}: {len(cfg['features']):>2} variables | {len(cfg['variables_log']):>2} con log1p | {cfg['nombre']}")

  🎯 LOS 4 MODELOS QUE SE EXPORTAN
 • Modelo 1:  5 variables |  0 con log1p | Modelo 1 · con fuga (HandHQ, 5 variables)
 • Modelo 2:  5 variables |  0 con log1p | Modelo 2 · corregido (HandHQ, 5 variables)
 • Modelo 3: 44 variables |  7 con log1p | Modelo 3 · 44 variables públicas (HandHQ)
 • Modelo 4: 47 variables |  7 con log1p | Modelo 4 · Pluribus (47 variables con cartas)


In [4]:
def preparar_crudo(df, cfg):
    """
    Variables en el orden del modelo, log1p (con piso en 0) a las cantidades de dinero.
    Es la misma preparación de los Notebooks 04, 07, 09 y 11; devuelve la matriz ANTES de escalar.
    """
    X = df[cfg['features']].astype('float64').copy()
    for c in cfg['variables_log']:
        X[c] = np.log1p(X[c].clip(lower=0))
    X = X.values
    return X.astype('float32') if cfg['pre_float32'] else X

def escalar(X, escalador):
    """Aplica el StandardScaler del .pkl y entrega float32, que es lo que reciben Keras y ONNX."""
    if hasattr(escalador, 'feature_names_in_'):          # Modelos 1 y 2 se ajustaron con un DataFrame
        X = pd.DataFrame(X, columns=list(escalador.feature_names_in_))
    return np.asarray(escalador.transform(X)).astype('float32')

print("✅ Funciones de preparación definidas (log1p con piso en 0 ➔ escalador ➔ float32)")

✅ Funciones de preparación definidas (log1p con piso en 0 ➔ escalador ➔ float32)


In [5]:
D = {}   # todo lo de cada modelo: red, escalador, test, matrices y probabilidades de Keras
print("=" * 65)
print("  📖 CARGA DE MODELOS, ESCALADORES Y CONJUNTOS DE PRUEBA")
print("=" * 65)
for m, cfg in MODELOS.items():
    t0 = time.time()
    modelo = keras.models.load_model(f"../models/{cfg['carpeta']}/{cfg['keras']}")
    escalador = joblib.load(f"../models/{cfg['carpeta']}/{cfg['scaler']}")
    df_test = pd.read_parquet(cfg['test'])
    X_pre = preparar_crudo(df_test, cfg)
    X_esc = escalar(X_pre, escalador)
    y_test = df_test['accion'].values
    prob_keras = modelo.predict(X_esc, batch_size=8192, verbose=0)
    D[m] = dict(modelo=modelo, escalador=escalador, df_test=df_test, X_pre=X_pre, X_esc=X_esc, y=y_test, prob_keras=prob_keras)
    acc = (prob_keras.argmax(axis=1) == y_test).mean() * 100
    print(f" • Modelo {m}: {len(df_test):>9,} decisiones de test | entrada {X_esc.shape[1]:>2} | exactitud Keras {acc:.2f}% | {time.time() - t0:.0f} s")

  📖 CARGA DE MODELOS, ESCALADORES Y CONJUNTOS DE PRUEBA


 • Modelo 1:    56,362 decisiones de test | entrada  5 | exactitud Keras 92.90% | 1 s


 • Modelo 2:    56,362 decisiones de test | entrada  5 | exactitud Keras 64.86% | 1 s


 • Modelo 3:   593,212 decisiones de test | entrada 44 | exactitud Keras 73.83% | 3 s


 • Modelo 4:    16,599 decisiones de test | entrada 47 | exactitud Keras 87.39% | 1 s


In [6]:
INFO_ONNX = {}
print("=" * 65)
print(f"  💾 EXPORTACIÓN A ONNX (opset {OPSET}, modo inferencia)")
print("=" * 65)
for m, cfg in MODELOS.items():
    ruta = os.path.join(SALIDA, f"modelo{m}.onnx")
    modelo = D[m]['modelo']

    # 1. Keras exige haber llamado al modelo una vez; training=False deja Dropout apagado y BatchNorm con lo aprendido
    modelo(np.zeros((1, len(cfg['features'])), dtype='float32'), training=False)

    # 2. Exportar con Keras 3 (usa tf2onnx por dentro)
    modelo.export(ruta, format="onnx", opset_version=OPSET)

    # 3. Revisar el archivo: validez, versión de opset, nombres de entrada/salida y operaciones usadas
    grafo = onnx.load(ruta)
    onnx.checker.check_model(grafo)
    opset = {o.domain or 'ai.onnx': o.version for o in grafo.opset_import}['ai.onnx']
    forma = lambda t: [d.dim_value if d.dim_value else 'N' for d in t.type.tensor_type.shape.dim]
    entrada, salida = grafo.graph.input[0], grafo.graph.output[0]
    operaciones = sorted(set(n.op_type for n in grafo.graph.node))
    INFO_ONNX[m] = dict(entrada=entrada.name, forma_entrada=forma(entrada), salida=salida.name,
                        forma_salida=forma(salida), opset=int(opset), operaciones=operaciones)
    assert opset == OPSET, f"Modelo {m}: opset {opset} distinto de {OPSET}"
    assert 'Dropout' not in operaciones and 'BatchNormalization' not in operaciones, "Quedaron capas de entrenamiento"
    print(f" • modelo{m}.onnx: {os.path.getsize(ruta) / 1024:.1f} KB | opset {opset} | entrada '{entrada.name}' {forma(entrada)} "
          f"| salida '{salida.name}' {forma(salida)}")
    print(f"     operaciones: {', '.join(operaciones)}")
print("\n✅ Los 4 modelos exportados; sin Dropout ni BatchNormalization sueltos (ya integrados en las capas densas)")

  💾 EXPORTACIÓN A ONNX (opset 17, modo inferencia)


C:\Users\lucas\miniconda3\envs\ia_local\Lib\site-packages\keras\src\export\tf2onnx_lib.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Saved artifact at '../app/modelo_transformado\modelo1.onnx'.


 • modelo1.onnx: 48.6 KB | opset 17 | entrada 'Entrada_Variables' ['N', 5] | salida 'Identity:0' ['N', 3]
     operaciones: Add, MatMul, Mul, Relu, Softmax


Saved artifact at '../app/modelo_transformado\modelo2.onnx'.


 • modelo2.onnx: 48.7 KB | opset 17 | entrada 'Entrada_Variables' ['N', 5] | salida 'Identity:0' ['N', 3]
     operaciones: Add, MatMul, Mul, Relu, Softmax


Saved artifact at '../app/modelo_transformado\modelo3.onnx'.


 • modelo3.onnx: 213.4 KB | opset 17 | entrada 'Entrada_Variables' ['N', 44] | salida 'Identity:0' ['N', 3]
     operaciones: Add, MatMul, Mul, Relu, Softmax


Saved artifact at '../app/modelo_transformado\modelo4.onnx'.


 • modelo4.onnx: 216.4 KB | opset 17 | entrada 'Entrada_Variables' ['N', 47] | salida 'Identity:0' ['N', 3]
     operaciones: Add, MatMul, Mul, Relu, Softmax

✅ Los 4 modelos exportados; sin Dropout ni BatchNormalization sueltos (ya integrados en las capas densas)


In [7]:
UMBRAL_PROB = 1e-5     # prueba 1
UMBRAL_ESC = 1e-6      # prueba 4

def correr_onnx(ruta, X, lote=65536):
    """Ejecuta un .onnx con onnxruntime (CPU) por lotes y devuelve las probabilidades."""
    sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])
    nombre = sesion.get_inputs()[0].name
    return np.concatenate([sesion.run(None, {nombre: X[i:i + lote]})[0] for i in range(0, len(X), lote)])

R = {}      # resultados de las pruebas 1 a 4
fallos = []
print("=" * 78)
print("  🛡️ PRUEBAS 1, 2 y 3 · ONNX contra Keras sobre el test COMPLETO de cada modelo")
print("=" * 78)
for m in MODELOS:
    d = D[m]
    d['prob_onnx'] = correr_onnx(os.path.join(SALIDA, f"modelo{m}.onnx"), d['X_esc'])
    pk, po, y = d['prob_keras'], d['prob_onnx'], d['y']
    dif_max = float(np.abs(pk - po).max())
    coincide = float((pk.argmax(axis=1) == po.argmax(axis=1)).mean() * 100)
    acc_k = float((pk.argmax(axis=1) == y).mean() * 100)
    acc_o = float((po.argmax(axis=1) == y).mean() * 100)
    R[m] = dict(filas=len(y), dif_max=dif_max, coincidencia=coincide, acc_keras=acc_k, acc_onnx=acc_o)
    ok1, ok2, ok3 = dif_max <= UMBRAL_PROB, coincide == 100.0, acc_k == acc_o
    print(f"\n Modelo {m} ({len(y):,} decisiones)")
    print(f"   1. Diferencia máxima de probabilidad : {dif_max:.2e}  (criterio ≤ {UMBRAL_PROB:.0e})  {'✅' if ok1 else '❌'}")
    print(f"   2. Coincidencia de la acción elegida : {coincide:.4f}%  (criterio 100%)  {'✅' if ok2 else '❌'}")
    print(f"   3. Exactitud Keras {acc_k:.2f}% | ONNX {acc_o:.2f}%  (documentada: {MODELOS[m]['ref_exactitud']:.2f}%)  {'✅' if ok3 else '❌'}")
    if not (ok1 and ok2 and ok3):
        fallos.append(f"Modelo {m}")

# 2. Criterio duro del Modelo 4: debe reproducir la exactitud del Notebook 11
ok_m4 = round(R[4]['acc_onnx'], 2) == 87.39
print(f"\n 🎯 Modelo 4 con ONNX: {R[4]['acc_onnx']:.2f}% (Notebook 11: 87.39%) {'✅' if ok_m4 else '❌'}")
if not ok_m4:
    fallos.append("Exactitud del Modelo 4")
assert not fallos, f"Pruebas fallidas: {fallos}"
print("=" * 78)

  🛡️ PRUEBAS 1, 2 y 3 · ONNX contra Keras sobre el test COMPLETO de cada modelo

 Modelo 1 (56,362 decisiones)
   1. Diferencia máxima de probabilidad : 9.54e-07  (criterio ≤ 1e-05)  ✅
   2. Coincidencia de la acción elegida : 100.0000%  (criterio 100%)  ✅
   3. Exactitud Keras 92.90% | ONNX 92.90%  (documentada: 92.90%)  ✅



 Modelo 2 (56,362 decisiones)
   1. Diferencia máxima de probabilidad : 4.47e-07  (criterio ≤ 1e-05)  ✅
   2. Coincidencia de la acción elegida : 100.0000%  (criterio 100%)  ✅
   3. Exactitud Keras 64.86% | ONNX 64.86%  (documentada: 64.86%)  ✅



 Modelo 3 (593,212 decisiones)
   1. Diferencia máxima de probabilidad : 2.68e-07  (criterio ≤ 1e-05)  ✅
   2. Coincidencia de la acción elegida : 100.0000%  (criterio 100%)  ✅
   3. Exactitud Keras 73.83% | ONNX 73.83%  (documentada: 73.83%)  ✅

 Modelo 4 (16,599 decisiones)
   1. Diferencia máxima de probabilidad : 8.05e-07  (criterio ≤ 1e-05)  ✅
   2. Coincidencia de la acción elegida : 100.0000%  (criterio 100%)  ✅
   3. Exactitud Keras 87.39% | ONNX 87.39%  (documentada: 87.39%)  ✅

 🎯 Modelo 4 con ONNX: 87.39% (Notebook 11: 87.39%) ✅


In [8]:
def pasos_preparacion(cfg):
    """Texto de los pasos que debe repetir la app antes de llamar al ONNX."""
    pasos = ["1. Tomar las variables en el orden de 'variables' (valores numéricos en float64)."]
    if cfg['variables_log']:
        pasos.append("2. A cada variable de 'variables_log' aplicarle x = log1p(max(x, 0)) (Math.log1p en JavaScript).")
    else:
        pasos.append("2. Este modelo no usa log1p: las variables entran tal cual.")
    if cfg['pre_float32']:
        pasos.append("3. Convertir a float32 (Math.fround) antes de escalar, como en el entrenamiento.")
    pasos.append("4. Escalar cada columna: z = (x - escalador.mean[i]) / escalador.scale[i].")
    pasos.append("5. Entregar un Float32Array de forma [N, n_variables] al ONNX; la salida son 3 probabilidades (softmax).")
    return pasos

print("=" * 78)
print("  💾 JSON DE PREPARACIÓN + 🛡️ PRUEBA 4 · escalador del JSON contra el .pkl")
print("=" * 78)
for m, cfg in MODELOS.items():
    d, esc, info = D[m], D[m]['escalador'], INFO_ONNX[m]
    prep = {
        "modelo": m,
        "nombre": cfg['nombre'],
        "origen_keras": f"models/{cfg['carpeta']}/{cfg['keras']}",
        "origen_escalador": f"models/{cfg['carpeta']}/{cfg['scaler']}",
        "archivo_onnx": f"modelo{m}.onnx",
        "onnx": {"opset": info['opset'],
                 "entrada": {"nombre": info['entrada'], "forma": info['forma_entrada'], "tipo": "float32"},
                 "salida": {"nombre": info['salida'], "forma": info['forma_salida'], "tipo": "float32"},
                 "descripcion_salida": "Probabilidades softmax (suman 1). La acción elegida es el índice del máximo."},
        "n_variables": len(cfg['features']),
        "variables": cfg['features'],
        "variables_log": cfg['variables_log'],
        "pasos_preparacion": pasos_preparacion(cfg),
        "escalador": {"tipo": "StandardScaler", "formula": "z = (x - mean) / scale",
                      "mean": [float(v) for v in esc.mean_], "scale": [float(v) for v in esc.scale_]},
        "clases_salida": NOMBRES_CLASES,
        "exactitud_test": {"conjunto": cfg['test'].replace('../', ''), "decisiones": R[m]['filas'],
                           "keras_pct": R[m]['acc_keras'], "onnx_pct": R[m]['acc_onnx']},
    }
    with open(os.path.join(SALIDA, f"modelo{m}_preparacion.json"), "w", encoding="utf-8") as f:
        json.dump(prep, f, ensure_ascii=False, indent=1)

    # PRUEBA 4: se vuelve a LEER el JSON del disco y se aplica a la matriz previa al escalado del test completo
    leido = cargar_json(os.path.join(SALIDA, f"modelo{m}_preparacion.json"))
    media, escala = np.array(leido['escalador']['mean']), np.array(leido['escalador']['scale'])
    X64 = d['X_pre'].astype('float64')
    z_json = (X64 - media) / escala                                     # lo que hará la app (float64)
    z_pkl = np.asarray(esc.transform(pd.DataFrame(X64, columns=list(esc.feature_names_in_))
                                     if hasattr(esc, 'feature_names_in_') else X64))
    dif_param = float(max(np.abs(media - esc.mean_).max(), np.abs(escala - esc.scale_).max()))
    dif_esc = float(np.abs(z_json - z_pkl).max())
    dif_f32 = float(np.abs(z_json.astype('float32') - d['X_esc']).max())  # informativo: ruta float32 de sklearn
    # Extra: probabilidades del ONNX alimentado con el escalado del JSON contra las de Keras
    prob_json = correr_onnx(os.path.join(SALIDA, f"modelo{m}.onnx"), z_json.astype('float32'))
    dif_prob_json = float(np.abs(prob_json - d['prob_keras']).max())
    R[m].update(dif_escalador=dif_esc, dif_parametros=dif_param, dif_escalador_f32=dif_f32, dif_prob_json=dif_prob_json)
    ok4 = dif_esc <= UMBRAL_ESC
    print(f" • Modelo {m}: parámetros JSON vs .pkl {dif_param:.1e} | 4. escalado JSON vs .pkl {dif_esc:.2e} "
          f"(criterio ≤ {UMBRAL_ESC:.0e}) {'✅' if ok4 else '❌'}")
    print(f"     informativo: contra la ruta float32 de sklearn {dif_f32:.2e} | ONNX con escalado del JSON vs Keras {dif_prob_json:.2e}")
    assert ok4, f"Modelo {m}: el escalador del JSON no coincide con el .pkl"
print("\n✅ Prueba 4 aprobada en los 4 modelos")

  💾 JSON DE PREPARACIÓN + 🛡️ PRUEBA 4 · escalador del JSON contra el .pkl


 • Modelo 1: parámetros JSON vs .pkl 0.0e+00 | 4. escalado JSON vs .pkl 0.00e+00 (criterio ≤ 1e-06) ✅
     informativo: contra la ruta float32 de sklearn 0.00e+00 | ONNX con escalado del JSON vs Keras 9.54e-07


 • Modelo 2: parámetros JSON vs .pkl 0.0e+00 | 4. escalado JSON vs .pkl 0.00e+00 (criterio ≤ 1e-06) ✅
     informativo: contra la ruta float32 de sklearn 0.00e+00 | ONNX con escalado del JSON vs Keras 4.47e-07


 • Modelo 3: parámetros JSON vs .pkl 0.0e+00 | 4. escalado JSON vs .pkl 0.00e+00 (criterio ≤ 1e-06) ✅
     informativo: contra la ruta float32 de sklearn 1.91e-06 | ONNX con escalado del JSON vs Keras 4.17e-07
 • Modelo 4: parámetros JSON vs .pkl 0.0e+00 | 4. escalado JSON vs .pkl 0.00e+00 (criterio ≤ 1e-06) ✅
     informativo: contra la ruta float32 de sklearn 3.81e-06 | ONNX con escalado del JSON vs Keras 1.25e-06

✅ Prueba 4 aprobada en los 4 modelos


In [9]:
# 1. La tabla que usó el Modelo 4 está en la columna c_eq_pre de sus datos (Notebook 10: caché de 169 manos × rivales)
RANGOS = '23456789TJQKA'
VALOR = {r: i for i, r in enumerate(RANGOS, start=2)}
LETRA = {v: r for r, v in VALOR.items()}

def clase_de_mano(carta1, carta2):
    """Convierte dos cartas ('9s', 'Js') en una de las 169 clases: 'J9s'. Pares 'TT'; suited 's'; offsuit 'o'; carta alta primero."""
    r1, r2 = sorted([carta1[0], carta2[0]], key=lambda r: VALOR[r], reverse=True)
    if r1 == r2:
        return r1 + r2
    return r1 + r2 + ('s' if carta1[1] == carta2[1] else 'o')

def separar_cartas(texto):
    """Convierte 'Qs9c2h' en ['Qs', '9c', '2h']."""
    return [texto[i:i + 2] for i in range(0, len(texto), 2)]

cols = ['hand_key', 'sesion', 'cartas', 'n_activos', 'calle', 'c_eq_pre']
df4_todo = pd.concat([pd.read_parquet(f"../data/processed/modelo4/{s}.parquet", columns=cols) for s in ['train', 'val', 'test']],
                     ignore_index=True)
df4_todo['clase'] = df4_todo['cartas'].map(lambda s: clase_de_mano(*separar_cartas(s)))
df4_todo['rivales'] = (df4_todo['n_activos'] - 1).clip(lower=1, upper=4)

# 2. Cada (clase, rivales) debe tener UN solo valor de equity en todas las decisiones
g = df4_todo.groupby(['clase', 'rivales'])['c_eq_pre']
unicos = g.nunique()
tabla_plana = g.first()
print("=" * 65)
print("  🎯 TABLA DE EQUITY PREFLOP DEL MODELO 4")
print("=" * 65)
print(f" • Decisiones revisadas (train+val+test): {len(df4_todo):,}")
print(f" • Clases de mano distintas: {df4_todo['clase'].nunique()} (esperado 169)")
print(f" • Combinaciones (clase, rivales): {len(tabla_plana)} (esperado 169 × 4 = 676)")
print(f" • Máximo de valores distintos por combinación: {unicos.max()} (debe ser 1)")
assert df4_todo['clase'].nunique() == 169 and len(tabla_plana) == 676 and unicos.max() == 1

# 3. Cada decisión (también las postflop) debe encontrar en la tabla EXACTAMENTE su c_eq_pre
tabla = {c: {str(r): float(tabla_plana[(c, r)]) for r in (1, 2, 3, 4)} for c in sorted(df4_todo['clase'].unique())}
buscado = np.array([tabla[c][str(r)] for c, r in zip(df4_todo['clase'], df4_todo['rivales'])])
dif_tabla = float(np.abs(buscado - df4_todo['c_eq_pre'].values).max())
print(f" • Diferencia máxima entre la tabla y c_eq_pre en las {len(df4_todo):,} decisiones: {dif_tabla:.1e} ✅")
assert dif_tabla == 0.0

# 4. Guardar el JSON con la explicación de las claves
equity_json = {
    "descripcion": "Equity preflop (probabilidad de ganar, empates cuentan la mitad) de cada una de las 169 manos iniciales "
                   "contra 1 a 4 rivales con manos aleatorias. Es EXACTAMENTE la tabla que usó el Modelo 4 como variable c_eq_pre.",
    "origen": "Notebook 10: equity_preflop() con 3,000 simulaciones Monte Carlo por celda (treys), semilla 42. "
              "Valores extraídos de la columna c_eq_pre de data/processed/modelo4/*.parquet y verificados en las "
              f"{len(df4_todo):,} decisiones.",
    "simulaciones_por_celda": 3000,
    "orden_rangos_de_menor_a_mayor": RANGOS,
    "clave_clase": "Dos rangos con la carta ALTA primero. Pares: 'AA', 'TT', '22'. Mismo palo: rangos + 's' ('AKs', 'J9s'). "
                   "Distinto palo: rangos + 'o' ('AKo', 'J9o'). Total: 13 pares + 78 suited + 78 offsuit = 169.",
    "clave_rivales": "Número de rivales = min(jugadores_activos - 1, 4), con mínimo 1 (jugadores_activos = n_activos de la situación, "
                     "incluido el que decide). Se escribe como texto: '1', '2', '3', '4'. Con 5 o más rivales se usa '4'.",
    "uso_en_el_modelo": "c_eq_pre = tabla[clase][rivales] en TODAS las calles. Preflop además c_equity = c_eq_pre.",
    "como_mapear_una_mano": {
        "pasos": ["1. Separar las dos cartas: '9sJs' ➔ '9s', 'Js'.",
                  "2. Ordenar los rangos de mayor a menor con 23456789TJQKA: J, 9.",
                  "3. Si los rangos son iguales ➔ par ('TT'). Si no, agregar 's' cuando los palos son iguales o 'o' cuando son distintos."],
        "ejemplos": [{"cartas": "9sJs", "clase": clase_de_mano('9s', 'Js')}, {"cartas": "Ad2s", "clase": clase_de_mano('Ad', '2s')},
                     {"cartas": "TsTc", "clase": clase_de_mano('Ts', 'Tc')}, {"cartas": "KhAh", "clase": clase_de_mano('Kh', 'Ah')}],
    },
    "n_clases": 169,
    "tabla": tabla,
}
with open(os.path.join(SALIDA, "equity_preflop_169.json"), "w", encoding="utf-8") as f:
    json.dump(equity_json, f, ensure_ascii=False, indent=1)
print(f"\n💾 equity_preflop_169.json guardado ({os.path.getsize(os.path.join(SALIDA, 'equity_preflop_169.json')) / 1024:.1f} KB)")
print(f" • Ejemplo '9sJs' ➔ clase '{clase_de_mano('9s', 'Js')}' ➔ contra 2 rivales: {tabla['J9s']['2']:.5f}")
print(f" • AA contra 1 rival: {tabla['AA']['1']:.4f} | 72o contra 1 rival: {tabla['72o']['1']:.4f}")

  🎯 TABLA DE EQUITY PREFLOP DEL MODELO 4
 • Decisiones revisadas (train+val+test): 91,356
 • Clases de mano distintas: 169 (esperado 169)
 • Combinaciones (clase, rivales): 676 (esperado 169 × 4 = 676)
 • Máximo de valores distintos por combinación: 1 (debe ser 1)


 • Diferencia máxima entre la tabla y c_eq_pre en las 91,356 decisiones: 0.0e+00 ✅

💾 equity_preflop_169.json guardado (21.2 KB)
 • Ejemplo '9sJs' ➔ clase 'J9s' ➔ contra 2 rivales: 0.39617
 • AA contra 1 rival: 0.8533 | 72o contra 1 rival: 0.3528


In [10]:
# 1. Reproducir EXACTAMENTE el orden del Notebook 10: hand_key = posición en sorted(glob('pluribus/*/*.phh'))
PLURIBUS = "../data/raw/pluribus"
archivos = sorted(glob.glob(os.path.join(PLURIBUS, "*", "*.phh")))
def sesion_y_archivo(ruta):
    """Devuelve ('100', '0.phh') a partir de la ruta del archivo."""
    return os.path.basename(os.path.dirname(ruta)), os.path.basename(ruta)

# 2. ¿Es lo mismo ordenar por (sesión, nombre de archivo) como texto? Así se documenta para la app
orden_tuplas = sorted(archivos, key=sesion_y_archivo)
print("=" * 65)
print("  🔑 CÓMO SE RELACIONA hand_key CON LOS ARCHIVOS PHH")
print("=" * 65)
print(f" • Archivos .phh encontrados: {len(archivos):,}")
print(f" • Orden por ruta completa == orden por (sesión, archivo) como texto: {archivos == orden_tuplas}")

# 3. Partición por sesión 70/15/15 (misma semilla y misma regla del Notebook 10)
sesiones = sorted(set(sesion_y_archivo(a)[0] for a in archivos))
np.random.default_rng(42).shuffle(sesiones)
c_train, c_val = int(len(sesiones) * 0.70), int(len(sesiones) * 0.85)
asignacion = {s: ('train' if i < c_train else 'val' if i < c_val else 'test') for i, s in enumerate(sesiones)}
claves_test = [k for k, a in enumerate(archivos) if asignacion[sesion_y_archivo(a)[0]] == 'test']

# 4. Contrastar con el parquet de test del Modelo 4
df4_test = D[4]['df_test']
claves_parquet = sorted(df4_test['hand_key'].unique())
sesiones_test = sorted(s for s, v in asignacion.items() if v == 'test')
print(f" • Sesiones de test (recalculadas): {len(sesiones_test)} | en el parquet: {df4_test['sesion'].nunique()} | iguales: "
      f"{sesiones_test == sorted(df4_test['sesion'].unique())}")
print(f" • Manos de las sesiones de test: {len(claves_test):,} | manos con decisiones en el parquet: {len(claves_parquet):,}")
assert claves_test == claves_parquet, "Las manos de las sesiones de test no coinciden con las del parquet"
assert all(sesion_y_archivo(archivos[k])[0] == s for k, s in zip(df4_test['hand_key'], df4_test['sesion']))
print("✅ hand_key ➔ archivo PHH verificado: todas las manos de test caen en la sesión correcta")

# 5. Leer las manos de test (formato TOML) como objetos sencillos
MANOS = {}
for k in claves_test:
    ses, nombre = sesion_y_archivo(archivos[k])
    with open(archivos[k], 'rb') as f:
        t = tomllib.load(f)
    with open(archivos[k], encoding='utf-8') as f:
        texto = f.read()
    MANOS[k] = SimpleNamespace(hand_key=k, sesion=ses, archivo=nombre, split='test', texto=texto,
                               players=t['players'], starting_stacks=t['starting_stacks'],
                               blinds_or_straddles=t['blinds_or_straddles'], actions=t['actions'],
                               finishing_stacks=t.get('finishing_stacks'), antes=t.get('antes'), min_bet=t.get('min_bet'))
print(f"📖 {len(MANOS):,} manos de test leídas de {len(sesiones_test)} sesiones")

  🔑 CÓMO SE RELACIONA hand_key CON LOS ARCHIVOS PHH
 • Archivos .phh encontrados: 10,000
 • Orden por ruta completa == orden por (sesión, archivo) como texto: True
 • Sesiones de test (recalculadas): 14 | en el parquet: 14 | iguales: True
 • Manos de las sesiones de test: 1,849 | manos con decisiones en el parquet: 1,849


✅ hand_key ➔ archivo PHH verificado: todas las manos de test caen en la sesión correcta


📖 1,849 manos de test leídas de 14 sesiones


In [11]:
evaluador = Evaluator()

def textura_mesa(board):
    """Textura de las cartas comunitarias visibles (misma función del Notebook 10)."""
    if not board:
        return dict(b_n=0, b_pareada=0, b_trio=0, b_max_palo=0, b_alta=0, b_broadway=0, b_conect=0)
    valores = [VALOR[c[0]] for c in board]
    palos = [c[1] for c in board]
    repeticiones = np.bincount(valores, minlength=15)
    unicos = set(valores) | ({1} if 14 in valores else set())
    return dict(b_n=len(board), b_pareada=int(repeticiones.max() >= 2), b_trio=int(repeticiones.max() >= 3),
                b_max_palo=max(palos.count(s) for s in set(palos)), b_alta=max(valores),
                b_broadway=sum(v >= 10 for v in valores),
                b_conect=max(sum(1 for k in range(lo, lo + 5) if k in unicos) for lo in range(1, 11)))

def variables_cartas_sin_mc(mis_cartas, mesa, num_rivales):
    """
    Variables de las cartas privadas, igual que el Notebook 10, con dos diferencias para poder VERIFICAR sin azar:
    c_eq_pre sale de la tabla guardada y c_equity (Monte Carlo) se deja en NaN.
    """
    valores = sorted([VALOR[c[0]] for c in mis_cartas], reverse=True)
    suited = mis_cartas[0][1] == mis_cartas[1][1]
    f = dict(c_alta=valores[0], c_baja=valores[1], c_pareja=int(valores[0] == valores[1]), c_suited=int(suited),
             c_gap=valores[0] - valores[1],
             c_eq_pre=tabla[clase_de_mano(*mis_cartas)][str(max(1, min(num_rivales, 4)))])
    if mesa:
        mias = [Card.new(c) for c in mis_cartas]
        rank = evaluador.evaluate([Card.new(c) for c in mesa], mias)
        todas = mis_cartas + mesa
        palos = [c[1] for c in todas]
        valores_todos = set(VALOR[c[0]] for c in todas) | ({1} if any(c[0] == 'A' for c in todas) else set())
        f.update(
            c_fuerza=1 - rank / 7462,
            c_clase=evaluador.get_rank_class(rank),
            c_proy_color=int(len(mesa) < 5 and any(palos.count(s) == 4 and s in (mis_cartas[0][1], mis_cartas[1][1]) for s in set(palos))),
            c_proy_escalera=int(len(mesa) < 5 and any(sum(k in valores_todos for k in range(lo, lo + 5)) == 4 for lo in range(1, 11))),
            c_equity=np.nan,
        )
    else:
        f.update(c_fuerza=0.0, c_clase=10, c_proy_color=0, c_proy_escalera=0, c_equity=f['c_eq_pre'])
    return f

def reconstruir_decisiones(mano):
    """
    Copia de reconstruir_decisiones_v4 (Notebook 10): máquina de estados mano a mano, foto de la mesa ANTES de actuar.
    Añade lo que necesita la app: el índice de cada acción dentro de la lista PHH y un registro de eventos
    (quién, qué, hasta cuánto, cuánto pagó, bote después) para el visor de manos.
    Devuelve (decisiones, eventos).
    """
    decisiones, eventos = [], []
    n = len(mano.players)
    ciegas = [float(c) for c in mano.blinds_or_straddles]
    bb = max(ciegas)
    pids = [f"p{i+1}" for i in range(n)]
    nombre = dict(zip(pids, mano.players))
    stacks = {pids[i]: float(mano.starting_stacks[i]) for i in range(n)}
    inicial = dict(stacks)
    puesto = {p: 0.0 for p in pids}
    bote, apuesta_alta = 0.0, 0.0
    for i, c in enumerate(ciegas):
        if c > 0:
            p = pids[i]; bote += c; stacks[p] -= c; puesto[p] = c; apuesta_alta = max(apuesta_alta, c)

    orden_pre = {p: (int(p[1:]) - 1 - 2) % n for p in pids}
    orden_post = {p: int(p[1:]) - 1 for p in pids}
    activos = set(pids)
    calle, board, cartas_privadas = 0, [], {}
    subidas_calle = callers_calle = checks_calle = subidas_pre = 0
    ultimo_agresor = agresor_prev = agresor_pre = None
    ult_apuesta = 0.0
    actuo_calle = {p: 0 for p in pids}
    subio_calle = {p: 0 for p in pids}

    for idx_act, act in enumerate(mano.actions):
        partes = act.strip().split()
        if not partes:
            continue
        actor = partes[0]

        # A. Acciones del crupier: reparto de cartas privadas ('dh') o comunitarias ('db')
        if actor == 'd':
            if len(partes) >= 4 and partes[1] == 'dh':
                cartas_privadas[partes[2]] = separar_cartas(partes[3])
            elif len(partes) >= 3 and partes[1] == 'db':
                calle += 1
                board += separar_cartas(partes[2])
                agresor_prev, ultimo_agresor = ultimo_agresor, None
                puesto = {p: 0.0 for p in pids}
                apuesta_alta = 0.0
                subidas_calle = callers_calle = checks_calle = 0
                ult_apuesta = 0.0
                actuo_calle = {p: 0 for p in pids}
                subio_calle = {p: 0 for p in pids}
            continue

        # B. Decide un jugador
        if len(partes) < 2 or partes[1] not in ('f', 'cc', 'cbr') or actor not in stacks:
            continue
        pid, tipo = actor, partes[1]

        # B1. FOTO DE LA MESA (antes de actuar)
        to_call = max(0.0, apuesta_alta - puesto[pid])
        stack = stacks[pid]
        rivales = [stacks[q] + puesto[q] for q in activos if q != pid]
        efectivo = min(stack + puesto[pid], max(rivales)) if rivales else stack
        orden = orden_pre if calle == 0 else orden_post
        detras = sum(1 for q in activos if q != pid and stacks[q] > 0 and orden[q] > orden[pid])
        label = {'f': 0, 'cc': 1, 'cbr': 2}[tipo]

        # B2. GUARDAR la foto, con las cartas PROPIAS
        mis_cartas = cartas_privadas[pid]
        fila = dict(
            hand_key=mano.hand_key, sesion=mano.sesion, jugador=nombre[pid], asiento=pid, indice_accion=idx_act,
            es_pluribus=int(nombre[pid] == 'Pluribus'), accion=label,
            cartas=''.join(mis_cartas), mesa=''.join(board),
            calle=calle, bote=bote, to_call=to_call, stack=stack,
            pot_odds=to_call / (bote + to_call) if bote + to_call > 0 else 0.0,
            n_activos=len(activos), n_allin=sum(1 for q in activos if stacks[q] <= 0), detras=detras,
            pos_pre=orden_pre[pid] / (n - 1), pos_post=orden_post[pid] / (n - 1),
            es_sb=int(pid == 'p1'), es_bb=int(pid == 'p2'), es_boton=int(pid == pids[-1]),
            bote_bb=bote / bb, to_call_bb=to_call / bb, stack_bb=stack / bb, eff_bb=efectivo / bb,
            spr=efectivo / bote if bote > 0 else 0.0,
            call_vs_bote=to_call / bote if bote > 0 else 0.0,
            call_es_allin=int(to_call > 0 and to_call >= stack),
            invertido_bb=(inicial[pid] - stack) / bb,
            subidas_calle=subidas_calle, callers_calle=callers_calle, checks_calle=checks_calle,
            subidas_pre=subidas_pre, ult_apuesta_vs_bote=ult_apuesta,
            ya_actuo_calle=actuo_calle[pid], ya_subio_calle=subio_calle[pid],
            es_agresor_pre=int(agresor_pre == pid), es_agresor_prev=int(agresor_prev == pid),
            hay_agresor_prev=int(agresor_prev is not None),
            **textura_mesa(board),
            **variables_cartas_sin_mc(mis_cartas, board, len(activos) - 1),
        )
        decisiones.append(fila)

        # B3. SOLO DESPUÉS aplicamos la acción a la mesa
        hasta = 0.0
        actuo_calle[pid] += 1
        if tipo == 'f':
            activos.discard(pid)
            pago = 0.0
        elif tipo == 'cc':
            pago = min(to_call, stack)
            bote += pago; stacks[pid] -= pago; puesto[pid] += pago
            if to_call > 0:
                callers_calle += 1
            else:
                checks_calle += 1
        else:
            total = float(partes[2]) if len(partes) >= 3 else apuesta_alta
            hasta = total
            pago = min(max(0.0, total - puesto[pid]), stack)
            ult_apuesta = pago / bote if bote > 0 else 0.0
            bote += pago; stacks[pid] -= pago; puesto[pid] += pago
            apuesta_alta = max(apuesta_alta, puesto[pid])
            subidas_calle += 1; subio_calle[pid] += 1
            ultimo_agresor = pid
            if calle == 0:
                subidas_pre += 1; agresor_pre = pid
        eventos.append(dict(idx=idx_act, pid=pid, tipo=tipo, hasta=hasta, pago=pago, bote=bote, calle=calle))
    return decisiones, eventos

# 1. Reconstruir TODAS las manos de test (sin Monte Carlo: es rápido)
t0 = time.time()
filas_rec, EVENTOS = [], {}
for k in claves_test:
    dec, ev = reconstruir_decisiones(MANOS[k])
    assert len(dec) == len(ev)
    filas_rec.extend(dec)
    EVENTOS[k] = ev
df_rec = pd.DataFrame(filas_rec)
print(f"⚙️ {len(MANOS):,} manos reconstruidas ➔ {len(df_rec):,} decisiones en {time.time() - t0:.1f} s")

# 2. Comparar con el parquet de test: mismas filas, mismo orden y las 46 variables (todas menos c_equity) IDÉNTICAS
assert len(df_rec) == len(df4_test), "Distinta cantidad de decisiones que el parquet"
iguales_id = all((df_rec[c].astype(str).values == df4_test[c].astype(str).values).all()
                 for c in ['hand_key', 'sesion', 'jugador', 'cartas', 'mesa'])
FEATURES4 = MODELOS[4]['features']
sin_mc = [c for c in FEATURES4 if c != 'c_equity']
dif_cols = {c: float(np.abs(df_rec[c].values.astype('float64') - df4_test[c].values.astype('float64')).max()) for c in sin_mc}
mal = {c: v for c, v in dif_cols.items() if v != 0.0}
acc_igual = bool((df_rec['accion'].values == df4_test['accion'].values).all())
preflop = df4_test['calle'].values == 0
eq_pre_igual = bool((df_rec.loc[preflop, 'c_equity'].values == df4_test.loc[preflop, 'c_equity'].values).all())
print("=" * 65)
print("  🛡️ RECONSTRUCCIÓN DESDE LOS ARCHIVOS PHH vs PARQUET DE TEST")
print("=" * 65)
print(f" • Manos, sesiones, jugadores, cartas y mesa idénticos: {iguales_id}")
print(f" • Acción real idéntica en las {len(df_rec):,} decisiones: {acc_igual}")
print(f" • Variables sin Monte Carlo idénticas (46 de 47): {len(mal) == 0} | columnas con diferencia: {mal if mal else 'ninguna'}")
print(f" • c_equity preflop (= c_eq_pre) idéntica: {eq_pre_igual}")
assert iguales_id and acc_igual and not mal and eq_pre_igual
print("✅ El orden de decisiones y los datos crudos de cada mano bastan para recalcular las variables desde cero")

⚙️ 1,849 manos reconstruidas ➔ 16,599 decisiones en 1.4 s
  🛡️ RECONSTRUCCIÓN DESDE LOS ARCHIVOS PHH vs PARQUET DE TEST
 • Manos, sesiones, jugadores, cartas y mesa idénticos: True
 • Acción real idéntica en las 16,599 decisiones: True
 • Variables sin Monte Carlo idénticas (46 de 47): True | columnas con diferencia: ninguna
 • c_equity preflop (= c_eq_pre) idéntica: True
✅ El orden de decisiones y los datos crudos de cada mano bastan para recalcular las variables desde cero


In [12]:
def f32_corto(v):
    """float32 ➔ número JSON que, al leerlo y pasarlo a float32 (Math.fround), da EXACTAMENTE el mismo float32."""
    return float(str(np.float32(v)))

def entero_o_float(v):
    """Valores numéricos del parquet sin perder precisión: enteros como enteros, el resto como float de 64 bits."""
    return int(v) if float(v).is_integer() and not isinstance(v, (float, np.floating)) else float(v)

# 1. ~1,000 situaciones del test del Modelo 4, estratificadas por calle (250 por calle, semilla 42)
POR_CALLE = 250
indices = []
for calle in range(4):
    pool = df4_test.index[df4_test['calle'] == calle]
    indices.extend(sorted(np.random.default_rng(42).choice(pool, size=min(POR_CALLE, len(pool)), replace=False)))
indices = sorted(int(i) for i in indices)

# 2. Posición de cada decisión: índice en la lista 'actions' del PHH y número de decisión dentro de la mano
orden_en_mano = df4_test.groupby('hand_key').cumcount().values
X4_esc, P4_keras = D[4]['X_esc'], D[4]['prob_keras']
situaciones = []
for n_sit, i in enumerate(indices):
    fila = df4_test.loc[i]
    rec = df_rec.loc[i]
    situaciones.append({
        "id": n_sit,
        "indice_test": i,
        "hand_key": int(fila['hand_key']),
        "indice_accion": int(rec['indice_accion']),
        "orden_decision": int(orden_en_mano[i]),
        "asiento": rec['asiento'],
        "jugador": fila['jugador'],
        "es_pluribus": int(fila['es_pluribus']),
        "cartas": fila['cartas'],
        "cartas_lista": separar_cartas(fila['cartas']),
        "mesa": fila['mesa'],
        "mesa_lista": separar_cartas(fila['mesa']),
        "calle": int(fila['calle']),
        "accion_real": int(fila['accion']),
        "variables": {c: entero_o_float(df4_test[c].values[i]) for c in FEATURES4},
        "vector_preparado": [f32_corto(v) for v in X4_esc[i]],
        "probabilidades_keras": [f32_corto(v) for v in P4_keras[i]],
    })

# 3. Datos crudos de cada mano referenciada (una sola vez): texto PHH + versión estructurada
claves_ref = sorted(set(s['hand_key'] for s in situaciones))
manos_ref = {}
for k in claves_ref:
    h = MANOS[k]
    manos_ref[str(k)] = {
        "archivo": f"{h.sesion}/{h.archivo}", "sesion": h.sesion,
        "texto_phh": h.texto,
        "jugadores": list(h.players), "asientos": [f"p{i + 1}" for i in range(len(h.players))],
        "stacks_iniciales": list(h.starting_stacks), "ciegas_o_straddles": list(h.blinds_or_straddles),
        "antes": list(h.antes) if h.antes is not None else None, "apuesta_minima": h.min_bet,
        "acciones": list(h.actions), "stacks_finales": list(h.finishing_stacks) if h.finishing_stacks else None,
    }

golden = {
    "metadatos": {
        "descripcion": "Situaciones reales del test del Modelo 4 para comprobar la app: variables crudas, vector preparado y probabilidades de Keras.",
        "modelo": "Modelo 4 · Pluribus (models/v4_pluribus)",
        "n_situaciones": len(situaciones), "situaciones_por_calle": {str(c): int((df4_test.loc[indices, 'calle'] == c).sum()) for c in range(4)},
        "semilla_muestreo": 42,
        "orden_variables": FEATURES4, "variables_log": MODELOS[4]['variables_log'],
        "clases_salida": NOMBRES_CLASES,
        "precision": "variables: float64 exacto (sin redondear). vector_preparado y probabilidades_keras: float32 escrito con su representación corta; "
                     "al leerlos y aplicar Math.fround dan exactamente el valor float32 de Python.",
        "preparacion": "vector_preparado = escalador( log1p(max(x,0)) en variables_log ), con la preparación de modelo4_preparacion.json.",
        "mapeo_hand_key_a_archivo_phh": {
            "regla": "hand_key = posición (desde 0) del archivo en sorted(glob('data/raw/pluribus/*/*.phh')), ordenado por la ruta completa como texto "
                     "(Notebook 10). Como no hay sesiones cuyo nombre sea prefijo de otro, equivale a ordenar por (carpeta de sesión, nombre de archivo) como texto.",
            "ejemplo": f"hand_key {claves_ref[0]} ➔ {manos_ref[str(claves_ref[0])]['archivo']}",
            "nota": "Cada mano en 'manos' trae su 'archivo' (sesion/nombre.phh) y el 'texto_phh' completo; la app no necesita reconstruir el orden.",
        },
        "indice_accion": "Posición (desde 0) de la decisión dentro de la lista 'acciones' de la mano (incluye las acciones 'd dh' y 'd db' del crupier). "
                         "'orden_decision' = número de decisión del jugador dentro de la mano (desde 0, solo f/cc/cbr).",
        "reglas_calculo": {
            "num_rivales_equity": "min(n_activos - 1, 4), mínimo 1",
            "c_equity": "Monte Carlo (150 simulaciones en el entrenamiento): NO es reproducible bit a bit; en la app se compara dentro del margen. "
                        "c_eq_pre sí es exacto: sale de equity_preflop_169.json.",
            "c_clase": "treys get_rank_class(puntaje): 0 = escalera real, 1 = escalera de color, ... 9 = carta alta; preflop = 10.",
        },
    },
    "manos": manos_ref,
    "situaciones": situaciones,
}
ruta_golden = os.path.join(SALIDA, "referencia_golden.json")
with open(ruta_golden, "w", encoding="utf-8") as f:
    json.dump(golden, f, ensure_ascii=False, separators=(',', ':'))
print("=" * 65)
print("  💾 REFERENCIA GOLDEN GUARDADA")
print("=" * 65)
print(f" • Situaciones: {len(situaciones):,} | por calle: {golden['metadatos']['situaciones_por_calle']}")
print(f" • Manos referenciadas: {len(manos_ref):,}")
print(f" • Archivo: {ruta_golden} ➔ {os.path.getsize(ruta_golden) / 1024**2:.2f} MB")

  💾 REFERENCIA GOLDEN GUARDADA
 • Situaciones: 1,000 | por calle: {'0': 250, '1': 250, '2': 250, '3': 250}
 • Manos referenciadas: 676
 • Archivo: ../app/modelo_transformado\referencia_golden.json ➔ 2.32 MB


In [13]:
# 1. Releer el JSON del disco y compararlo con el parquet y con una re-preparación independiente
leido = cargar_json(os.path.join(SALIDA, "referencia_golden.json"))
cfg4, esc4 = MODELOS[4], D[4]['escalador']
n_vars_malas = n_vec_malos = n_prob_malas = n_fila_mala = n_phh_mal = 0
for s in leido['situaciones']:
    fila = df4_test.loc[s['indice_test']]
    # 1a. Las 47 variables del JSON == las del parquet (igualdad exacta)
    n_vars_malas += sum(s['variables'][c] != fila[c] for c in cfg4['features'])
    # 1b. Mismas cartas, mesa, jugador y acción
    n_fila_mala += (s['cartas'], s['mesa'], s['jugador'], s['accion_real'], s['hand_key']) != (fila['cartas'], fila['mesa'], fila['jugador'], int(fila['accion']), int(fila['hand_key']))
    # 1c. Vector preparado del JSON == preparar_crudo + escalar sobre las variables del JSON
    crudo = preparar_crudo(pd.DataFrame([s['variables']]), cfg4)
    vec = escalar(crudo, esc4)[0]
    n_vec_malos += int(not np.array_equal(np.float32(s['vector_preparado']), vec))
    # 1d. Probabilidades de Keras guardadas == recalculadas
    n_prob_malas += int(not np.array_equal(np.float32(s['probabilidades_keras']), D[4]['prob_keras'][s['indice_test']]))
    # 1e. La acción indicada del texto PHH es de ese asiento y del tipo correcto
    acts = leido['manos'][str(s['hand_key'])]['acciones']
    p = acts[s['indice_accion']].split()
    n_phh_mal += int(p[0] != s['asiento'] or {'f': 0, 'cc': 1, 'cbr': 2}[p[1]] != s['accion_real'])
print("=" * 65)
print("  🛡️ VERIFICACIÓN DEL ARCHIVO GOLDEN (releído del disco)")
print("=" * 65)
print(f" • Variables del JSON distintas al parquet (de {len(leido['situaciones']) * len(cfg4['features']):,}): {n_vars_malas}")
print(f" • Situaciones con cartas/mesa/jugador/acción distintos: {n_fila_mala}")
print(f" • Vectores preparados distintos al recalcularlos: {n_vec_malos}")
print(f" • Probabilidades de Keras distintas: {n_prob_malas}")
print(f" • Índices de acción que no coinciden con el texto PHH: {n_phh_mal}")
assert n_vars_malas == n_fila_mala == n_vec_malos == n_prob_malas == n_phh_mal == 0
print("✅ El golden coincide exactamente con el parquet y con el PHH")

  🛡️ VERIFICACIÓN DEL ARCHIVO GOLDEN (releído del disco)
 • Variables del JSON distintas al parquet (de 47,000): 0
 • Situaciones con cartas/mesa/jugador/acción distintos: 0
 • Vectores preparados distintos al recalcularlos: 0
 • Probabilidades de Keras distintas: 0
 • Índices de acción que no coinciden con el texto PHH: 0
✅ El golden coincide exactamente con el parquet y con el PHH


In [14]:
def limpiar(v, decimales=5):
    """Redondea un valor DE VISUALIZACIÓN; los enteros se escriben sin decimales (el golden nunca se redondea)."""
    v = round(float(v), decimales)
    return int(v) if v == int(v) else v

def entero(v):
    """Fichas como entero cuando son exactas."""
    return int(v) if float(v).is_integer() else float(v)

CODIGO = {'f': 0, 'cc': 1, 'cbr': 2}
X4_crudo = df4_test[FEATURES4].values.astype('float64')
P4_onnx = D[4]['prob_onnx']
posiciones = df4_test.groupby('hand_key').indices          # filas del test de cada mano, en orden de decisión

# 1. Armar cada mano de forma compacta
banco = []
for k in claves_test:
    h = MANOS[k]
    hole, calles = [None] * len(h.players), []
    for act in h.actions:
        p = act.split()
        if p[:2] == ['d', 'dh']:
            hole[int(p[2][1:]) - 1] = p[3]
        elif p[:2] == ['d', 'db']:
            calles.append(p[2])
    ev, pos = EVENTOS[k], posiciones[k]
    a = [[int(e['pid'][1:]) - 1, CODIGO[e['tipo']], entero(e['hasta']), entero(e['pago']), e['calle'], entero(e['bote'])] for e in ev]
    d = [[i, int(df4_test['accion'].values[p]), [limpiar(x) for x in X4_crudo[p]], [limpiar(x) for x in P4_onnx[p]]]
         for i, p in enumerate(pos)]
    banco.append({"k": k, "s": h.sesion, "f": f"{h.sesion}/{h.archivo}", "n": list(h.players),
                  "st": [entero(x) for x in h.starting_stacks], "bl": [entero(x) for x in h.blinds_or_straddles],
                  "fin": [entero(x) for x in h.finishing_stacks] if h.finishing_stacks else None,
                  "h": hole, "t": calles, "a": a, "d": d})

# 2. Metadatos y explicación de las claves cortas
meta = {
    "fuente": "real", "descripcion": "Manos reales de Pluribus (6 jugadores, 100 ciegas grandes) del conjunto de TEST del Modelo 4",
    "origen": "Brown & Sandholm (Science, 2019); manos en formato PHH del repositorio uoftcprg/phh-dataset",
    "licencia": "MIT",
    "n_manos": len(banco), "n_decisiones": int(sum(len(m['d']) for m in banco)),
    "sesiones": sesiones_test,
    "orden_variables": FEATURES4, "clases_salida": NOMBRES_CLASES,
    "redondeo": "Datos de visualización redondeados a 5 decimales (variables y probabilidades). NO se usan para comprobar la app: eso es referencia_golden.json.",
    "probabilidades": "Salida del modelo4.onnx (onnxruntime) para la misma situación.",
    "claves_mano": {"k": "hand_key", "s": "sesión", "f": "archivo PHH (sesion/nombre)", "n": "nombres por asiento (asiento 0 = p1 = ciega pequeña)",
                    "st": "stacks iniciales en fichas", "bl": "ciegas o straddles por asiento (50/100)", "fin": "stacks finales",
                    "h": "cartas privadas por asiento (ej. 'TcQc')", "t": "cartas comunitarias por calle: [flop, turn, river] (ej. ['7d5h9d','7c','Qh'])",
                    "a": "acciones de los jugadores en orden: [asiento, código, hasta, pago, calle, bote_después]. código 0=fold, 1=check/call, 2=bet/raise. "
                         "'hasta' = total al que se sube (solo bet/raise, como en PHH 'cbr X'); 'pago' = fichas puestas con esa acción; calle 0..3",
                    "d": "decisiones: [índice en 'a', acción real (0/1/2), 47 variables en orden_variables, 3 probabilidades del Modelo 4]"},
}
contenido = json.dumps({"meta": meta, "manos": banco}, ensure_ascii=False, separators=(',', ':'))
tam = len(contenido.encode('utf-8'))
LIMITE = 8 * 1024 ** 2
ruta_banco, carpeta_banco = os.path.join(SALIDA, "banco_manos_reales.json"), os.path.join(SALIDA, "banco")

print("=" * 65)
print("  💾 BANCO DE MANOS REALES PARA EL VISOR")
print("=" * 65)
print(f" • Manos: {meta['n_manos']:,} | decisiones: {meta['n_decisiones']:,} | tamaño en un solo archivo: {tam / 1024**2:.2f} MB")

# 3. Un solo archivo si cabe; si no, un archivo por sesión con un índice
if tam <= LIMITE:
    if os.path.isdir(carpeta_banco):
        shutil.rmtree(carpeta_banco)
    with open(ruta_banco, "w", encoding="utf-8") as f:
        f.write(contenido)
    print(f" ✅ Cabe en un solo archivo (≤ 8 MB): {ruta_banco} ➔ {os.path.getsize(ruta_banco) / 1024**2:.2f} MB")
else:
    if os.path.exists(ruta_banco):
        os.remove(ruta_banco)
    shutil.rmtree(carpeta_banco, ignore_errors=True)
    os.makedirs(carpeta_banco)
    indice = []
    for s in sesiones_test:
        manos_s = [m for m in banco if m['s'] == s]
        nombre = f"sesion_{s}.json"
        with open(os.path.join(carpeta_banco, nombre), "w", encoding="utf-8") as f:
            json.dump({"sesion": s, "manos": manos_s}, f, ensure_ascii=False, separators=(',', ':'))
        indice.append({"sesion": s, "archivo": f"banco/{nombre}", "manos": len(manos_s),
                       "decisiones": sum(len(m['d']) for m in manos_s), "bytes": os.path.getsize(os.path.join(carpeta_banco, nombre))})
    with open(os.path.join(SALIDA, "banco_indice.json"), "w", encoding="utf-8") as f:
        json.dump({"meta": meta, "sesiones": indice}, f, ensure_ascii=False, separators=(',', ':'))
    print(f" ⚠️ Supera 8 MB: dividido en {len(indice)} archivos por sesión en banco/ + banco_indice.json")
    for it in indice:
        print(f"    • {it['archivo']}: {it['manos']:,} manos | {it['bytes'] / 1024**2:.2f} MB")

  💾 BANCO DE MANOS REALES PARA EL VISOR
 • Manos: 1,849 | decisiones: 16,599 | tamaño en un solo archivo: 3.41 MB
 ✅ Cabe en un solo archivo (≤ 8 MB): ../app/modelo_transformado\banco_manos_reales.json ➔ 3.41 MB


In [15]:
# 1. Evaluador: 3,000 manos al azar (1,000 de 5, 6 y 7 cartas), semilla fija
BARAJA = [r + s for r in RANGOS for s in 'shdc']
rng_eval = random.Random(2026)
manos_eval = []
for n_cartas in (5, 6, 7):
    for _ in range(1000):
        cartas = rng_eval.sample(BARAJA, n_cartas)
        enteros = [Card.new(c) for c in cartas]
        puntaje = evaluador.evaluate(enteros[:2], enteros[2:])
        manos_eval.append({"cartas": cartas, "puntaje": int(puntaje), "clase": int(evaluador.get_rank_class(puntaje))})

# 2. Control: el puntaje no depende del orden de las cartas
rng_orden = random.Random(7)
for m_ in manos_eval[::50]:
    mezcla = rng_orden.sample(m_['cartas'], len(m_['cartas']))
    ints = [Card.new(c) for c in mezcla]
    assert evaluador.evaluate(ints[:2], ints[2:]) == m_['puntaje']

prueba_evaluador = {
    "descripcion": "3,000 manos con el puntaje de treys (Evaluator().evaluate) para comprobar el evaluador de la app.",
    "semilla": 2026, "version_treys": "0.1.8",
    "formato": "cartas: texto 'As', 'Td' (rango 23456789TJQKA + palo s/h/d/c). puntaje: menor = más fuerte (1 = escalera real, 7462 = la peor). clase: get_rank_class(puntaje).",
    "cantidad_por_numero_de_cartas": {"5": 1000, "6": 1000, "7": 1000},
    "notas_treys": {
        "codificacion_carta": "Card.new('As') devuelve un entero de 32 bits: xxxbbbbb bbbbbbbb cdhsrrrr xxpppppp. "
                              "b = bit del rango (1 << rango), cdhs = palo en 4 bits (s=1, h=2, d=4, c=8, guardado en los bits 12-15), "
                              "r = rango 0..12 (2=0, ..., A=12) en los bits 8-11, p = primo del rango en los bits 0-5.",
        "primos_por_rango": dict(zip(RANGOS, Card.PRIMES)),
        "palo_a_entero": Card.CHAR_SUIT_TO_INT_SUIT,
        "como_evalua": "5 cartas: si las 5 comparten palo se busca el OR de los bits de rango en la tabla de colores (flush_lookup); "
                       "si no, se busca el producto de los primos en la tabla unsuited_lookup (parejas, tríos, escaleras, carta alta). "
                       "6 y 7 cartas: se evalúan todas las combinaciones de 5 y se toma el MENOR puntaje.",
        "umbrales_de_puntaje": {"escalera_real": LookupTable.MAX_ROYAL_FLUSH, "escalera_de_color": LookupTable.MAX_STRAIGHT_FLUSH,
                                "poker": LookupTable.MAX_FOUR_OF_A_KIND, "full_house": LookupTable.MAX_FULL_HOUSE,
                                "color": LookupTable.MAX_FLUSH, "escalera": LookupTable.MAX_STRAIGHT,
                                "trio": LookupTable.MAX_THREE_OF_A_KIND, "doble_par": LookupTable.MAX_TWO_PAIR,
                                "par": LookupTable.MAX_PAIR, "carta_alta": LookupTable.MAX_HIGH_CARD},
        "clases": {str(k): v for k, v in LookupTable.RANK_CLASS_TO_STRING.items()},
        "aviso_clase_cero": "En treys 0.1.8 get_rank_class devuelve 0 para el puntaje 1 (escalera real) y 1..9 para el resto; el Notebook 10 usa estos valores en c_clase.",
    },
    "manos": manos_eval,
}
ruta_eval = os.path.join(SALIDA, "prueba_evaluador.json")
with open(ruta_eval, "w", encoding="utf-8") as f:
    json.dump(prueba_evaluador, f, ensure_ascii=False, separators=(',', ':'))
clases_vistas = pd.Series([m_['clase'] for m_ in manos_eval]).value_counts().sort_index()
print("=" * 65)
print("  💾 PRUEBA DEL EVALUADOR DE MANOS")
print("=" * 65)
print(f" • Manos: {len(manos_eval):,} | puntaje mínimo {min(m_['puntaje'] for m_ in manos_eval):,} | máximo {max(m_['puntaje'] for m_ in manos_eval):,}")
print(" • Manos por clase:", {int(k): int(v) for k, v in clases_vistas.items()})
print(f" • Archivo: {ruta_eval} ➔ {os.path.getsize(ruta_eval) / 1024:.1f} KB")
print(f" • En el test del Modelo 4, c_clase va de {int(df4_test['c_clase'].min())} a {int(df4_test['c_clase'].max())} (10 = preflop)")

  💾 PRUEBA DEL EVALUADOR DE MANOS
 • Manos: 3,000 | puntaje mínimo 3 | máximo 7,461
 • Manos por clase: {1: 1, 2: 6, 3: 37, 4: 47, 5: 73, 6: 119, 7: 392, 8: 1329, 9: 996}
 • Archivo: ../app/modelo_transformado\prueba_evaluador.json ➔ 200.8 KB
 • En el test del Modelo 4, c_clase va de 1 a 10 (10 = preflop)


In [16]:
# 1. El Monte Carlo es lento en Python puro: se reparte entre los núcleos. Los procesos necesitan un módulo importable,
#    así que se escribe en una carpeta temporal (no queda nada en el proyecto).
CODIGO_TRABAJADOR = '''
import random
from treys import Card, Evaluator

_ev = Evaluator()
_MAZO = [Card.new(r + s) for r in "23456789TJQKA" for s in "shdc"]

def equity_monte_carlo(args):
    """Equity contra rivales con manos aleatorias (mismo método del Notebook 12). Devuelve victorias, empates y derrotas."""
    idx, mias, mesa, rivales, simulaciones, semilla = args
    rng = random.Random(semilla)
    mias_i = [Card.new(c) for c in mias]
    tab = [Card.new(c) for c in mesa]
    usadas = set(mias_i + tab)
    disponibles = [c for c in _MAZO if c not in usadas]
    faltan = 5 - len(tab)
    v = e = d = 0
    for _ in range(simulaciones):
        m = rng.sample(disponibles, faltan + 2 * rivales)
        mesa_final = tab + m[:faltan]
        mio = _ev.evaluate(mesa_final, mias_i)
        mejor = min(_ev.evaluate(mesa_final, m[faltan + 2 * k: faltan + 2 * k + 2]) for k in range(rivales))
        if mio < mejor:
            v += 1
        elif mio == mejor:
            e += 1
        else:
            d += 1
    return idx, v, e, d
'''
carpeta_trabajo = tempfile.mkdtemp(prefix="equity_pht_")
with open(os.path.join(carpeta_trabajo, "trabajador_equity.py"), "w", encoding="utf-8") as f:
    f.write(CODIGO_TRABAJADOR)
sys.path.insert(0, carpeta_trabajo)
import trabajador_equity
print(f"✅ Módulo de trabajo listo en una carpeta temporal ({os.cpu_count()} núcleos disponibles)")

✅ Módulo de trabajo listo en una carpeta temporal (20 núcleos disponibles)

In [17]:
from concurrent.futures import ProcessPoolExecutor

# 1. 200 situaciones: 50 por calle (0 a 3 cartas comunitarias 0/3/4/5), 1 a 4 rivales equilibrados, semilla fija
SIMULACIONES = 20000
SEMILLA_BASE = 2027
rng_eq = random.Random(SEMILLA_BASE)
situaciones_eq = []
for i in range(200):
    calle = i % 4
    n_mesa = {0: 0, 1: 3, 2: 4, 3: 5}[calle]
    rivales = (i // 4) % 4 + 1
    cartas = rng_eq.sample(BARAJA, 2 + n_mesa)
    situaciones_eq.append(dict(id=i, calle=calle, rivales=rivales, cartas=cartas[:2], mesa=cartas[2:]))

# 2. Calcular en paralelo (cada situación con su propia semilla ➔ resultado reproducible)
t0 = time.time()
tareas = [(s['id'], s['cartas'], s['mesa'], s['rivales'], SIMULACIONES, SEMILLA_BASE * 1000 + s['id']) for s in situaciones_eq]
with ProcessPoolExecutor(max_workers=min(os.cpu_count() or 2, 16)) as pool:
    resultados_eq = {r[0]: r for r in pool.map(trabajador_equity.equity_monte_carlo, tareas, chunksize=2)}
print(f"⚙️ {len(tareas)} situaciones × {SIMULACIONES:,} simulaciones en {time.time() - t0:.0f} s")

# 3. Equity y error estándar: cada simulación vale 1, 0.5 ó 0 ➔ varianza = E[x²] − E[x]²
salida_eq = []
for s in situaciones_eq:
    _, v, e, d = resultados_eq[s['id']]
    equity = (v + 0.5 * e) / SIMULACIONES
    varianza = (v + 0.25 * e) / SIMULACIONES - equity ** 2
    s['clase_mano'] = clase_de_mano(*s['cartas'])
    salida_eq.append({**s, "equity": equity, "error_estandar": float(np.sqrt(varianza / SIMULACIONES)),
                      "victorias": v, "empates": e, "derrotas": d})

# 4. Cordura: en preflop la tabla del Modelo 4 (3,000 simulaciones) debe coincidir dentro de su propio margen
z = []
for s in salida_eq:
    if s['calle'] == 0:
        ref = tabla[clase_de_mano(*s['cartas'])][str(s['rivales'])]
        se_tabla = np.sqrt(max(ref * (1 - ref), 0.01) / 3000)
        z.append((s['equity'] - ref) / np.sqrt(s['error_estandar'] ** 2 + se_tabla ** 2))
z = np.array(z)

prueba_equity = {
    "descripcion": "200 situaciones con equity calculado por Monte Carlo (20,000 simulaciones) para comprobar el motor de equity de la app.",
    "metodo": "Igual que el Notebook 12: se quitan las cartas conocidas de la baraja, a cada rival se le reparten 2 cartas al azar, se completa la mesa hasta 5 cartas "
              "y se compara con treys (menor puntaje gana). Equity = (victorias + empates/2) / simulaciones. Los rivales no se retiran ni se descartan manos.",
    "simulaciones": SIMULACIONES, "semilla_base": SEMILLA_BASE,
    "reproducibilidad": "Cada situación usa random.Random(semilla_base*1000 + id) con rng.sample sobre las cartas disponibles; no se pretende reproducirla bit a bit en TypeScript.",
    "error_estandar": "sqrt(varianza / simulaciones) con varianza = E[x²] − equity², x ∈ {0, 0.5, 1}. Para comparar con la app: |equity_ts − equity| ≤ 4 × sqrt(error_estandar² + error_estandar_ts²).",
    "n_situaciones": len(salida_eq),
    "formato": "cartas y mesa como texto 'As', 'Td'. calle 0=preflop, 1=flop, 2=turn, 3=river. rivales 1 a 4.",
    "situaciones": salida_eq,
}
ruta_equity = os.path.join(SALIDA, "prueba_equity.json")
with open(ruta_equity, "w", encoding="utf-8") as f:
    json.dump(prueba_equity, f, ensure_ascii=False, separators=(',', ':'))
se = np.array([s['error_estandar'] for s in salida_eq])
print("=" * 65)
print("  💾 PRUEBA DE EQUITY")
print("=" * 65)
print(f" • Situaciones: {len(salida_eq)} | por calle: { {c: sum(s['calle'] == c for s in salida_eq) for c in range(4)} } | por rivales: { {r: sum(s['rivales'] == r for s in salida_eq) for r in range(1, 5)} }")
print(f" • Error estándar: promedio {se.mean():.5f} | máximo {se.max():.5f}")
print(f" • Equity entre {min(s['equity'] for s in salida_eq):.3f} y {max(s['equity'] for s in salida_eq):.3f}")
print(f" • Cordura preflop contra la tabla del Modelo 4 ({len(z)} manos): |z| máximo {np.abs(z).max():.2f} | promedio de |z| {np.abs(z).mean():.2f} (esperado ≈ 0.8)")
print(f" • Archivo: {ruta_equity} ➔ {os.path.getsize(ruta_equity) / 1024:.1f} KB")
assert np.abs(z).max() < 4.5, "El Monte Carlo difiere de la tabla del Modelo 4 más de lo esperado"
sys.path.remove(carpeta_trabajo)
shutil.rmtree(carpeta_trabajo, ignore_errors=True)

⚙️ 200 situaciones × 20,000 simulaciones en 51 s
  💾 PRUEBA DE EQUITY
 • Situaciones: 200 | por calle: {0: 50, 1: 50, 2: 50, 3: 50} | por rivales: {1: 52, 2: 52, 3: 48, 4: 48}
 • Error estándar: promedio 0.00266 | máximo 0.00352
 • Equity entre 0.000 y 0.979
 • Cordura preflop contra la tabla del Modelo 4 (50 manos): |z| máximo 2.31 | promedio de |z| 0.65 (esperado ≈ 0.8)
 • Archivo: ../app/modelo_transformado\prueba_equity.json ➔ 40.0 KB


In [18]:
print("=" * 78)
print("  🎉 RESUMEN FINAL · NOTEBOOK 13")
print("=" * 78)
print("\n 🛡️ PRUEBAS 1 A 4 (conversión), test completo de cada modelo:")
print(f" {'Modelo':<8}{'Decisiones':>12}{'Dif. máx. prob.':>18}{'Coincidencia':>15}{'Exactitud Keras':>17}{'Exactitud ONNX':>16}{'Dif. escalador':>16}")
for m in MODELOS:
    r = R[m]
    print(f" {m:<8}{r['filas']:>12,}{r['dif_max']:>18.2e}{r['coincidencia']:>14.2f}%{r['acc_keras']:>16.2f}%{r['acc_onnx']:>15.2f}%{r['dif_escalador']:>16.2e}")

print("\n 💾 ARCHIVOS EN app/modelo_transformado/:")
total = 0
for raiz, _, nombres in os.walk(SALIDA):
    for nombre in sorted(nombres):
        if nombre.endswith(('.onnx', '.json')):
            ruta = os.path.join(raiz, nombre)
            tam = os.path.getsize(ruta)
            total += tam
            print(f"   • {os.path.relpath(ruta, SALIDA).replace(os.sep, '/'):<34}{tam / 1024:>12,.1f} KB")
print(f"   {'TOTAL':<36}{total / 1024**2:>10.2f} MB")

  🎉 RESUMEN FINAL · NOTEBOOK 13

 🛡️ PRUEBAS 1 A 4 (conversión), test completo de cada modelo:
 Modelo    Decisiones   Dif. máx. prob.   Coincidencia  Exactitud Keras  Exactitud ONNX  Dif. escalador
 1             56,362          9.54e-07        100.00%           92.90%          92.90%        0.00e+00
 2             56,362          4.47e-07        100.00%           64.86%          64.86%        0.00e+00
 3            593,212          2.68e-07        100.00%           73.83%          73.83%        0.00e+00
 4             16,599          8.05e-07        100.00%           87.39%          87.39%        0.00e+00

 💾 ARCHIVOS EN app/modelo_transformado/:
   • banco_manos_reales.json                3,487.2 KB
   • equity_preflop_169.json                   21.2 KB
   • modelo1.onnx                              48.6 KB
   • modelo1_preparacion.json                   1.7 KB
   • modelo2.onnx                              48.7 KB
   • modelo2_preparacion.json                   1.7 KB
   • modelo3.

### 🧠 Análisis Crítico

**Qué se hizo.** Los 4 modelos se pasaron de Keras a ONNX (opset 17, modo inferencia) con `model.export(..., format="onnx")` de Keras 3, que usa `tf2onnx` 1.17.0 por dentro. Funcionó con TensorFlow 2.21 y Keras 3.15 sin cambiar nada, siempre que `tf2onnx` se instale con `--no-deps` para no tocar numpy ni protobuf. Los grafos quedaron con solo `MatMul`, `Add`, `Mul`, `Relu` y `Softmax`: Dropout desaparece y BatchNormalization queda integrada en las capas densas. Entrada `Entrada_Variables` (N × variables) y salida `Identity:0` (N × 3 probabilidades: Fold, Check/Call, Bet/Raise).

**Resultados de las pruebas 1 a 4 (test completo de cada modelo).**
- Diferencia máxima de probabilidad ONNX vs Keras: 9.5e-07 (M1), 4.5e-07 (M2), 2.7e-07 (M3) y 8.1e-07 (M4). Todas están muy por debajo del límite de 1e-5.
- La acción elegida coincide en el 100% de las decisiones de los 4 modelos (56,362 · 56,362 · 593,212 · 16,599).
- La exactitud con ONNX es idéntica a la de Keras: 92.90%, 64.86%, 73.83% y **87.39%** en el Modelo 4, que coincide con el Notebook 11.
- El escalador guardado en JSON da 0.0 de diferencia con el `.pkl` cuando se aplica en float64, que es lo que hará la app.

**Qué se descubrió y conviene saber.**
- `variables_modelo3.json` y `variables_modelo4.json` listan `bote`, `to_call` y `stack` en `variables_log`, pero esas columnas no son entradas del modelo (solo sus versiones en ciegas). El código original las ignoraba; los JSON de preparación nuevos listan solo las 7 variables con `log1p` que de verdad se usan.
- Si el escalado se hace en float32, como en sklearn, aparece una diferencia de hasta 3.8e-6 con el cálculo en float64 en los Modelos 3 y 4 (redondeo de float32 en valores lejanos de la media). No afecta: con el escalado del JSON las probabilidades siguen a ≤ 1.3e-6 de Keras.
- La tabla de equity preflop tiene un único valor por cada una de las 676 combinaciones (169 manos × 1 a 4 rivales) y reproduce `c_eq_pre` sin diferencia en las 91,356 decisiones de entrenamiento, validación y test.
- Las manos de test se reconstruyeron desde los archivos PHH y las 46 variables sin Monte Carlo salen idénticas al parquet. Eso confirma que `hand_key` es la posición del archivo en el orden de rutas, y que los datos crudos del golden bastan para recalcular las variables desde cero en TypeScript. Solo `c_equity` postflop no se puede reproducir bit a bit: sale de 150 simulaciones con azar, y la app se compara dentro del margen del Monte Carlo.
- En el evaluador de treys 0.1.8, `get_rank_class` devuelve 0 para la escalera real, no 1 como decía un comentario antiguo.

**Límites.** Estas pruebas demuestran que el modelo convertido se comporta igual que el original; no prueban nada nuevo sobre la calidad del modelo. El Modelo 4 sigue describiendo lo que harían jugadores de élite en las manos de Pluribus, no una jugada óptima. Las pruebas 5 a 8 (variables, evaluador, equity y predicción dentro de la app) se harán con Vitest cuando exista el código, con los archivos `referencia_golden.json`, `prueba_evaluador.json` y `prueba_equity.json` generados aquí.
